##Imports



In [ ]:
import random
# import matplotlib.pyplot as plt
import heapq
# import numpy as np
import copy

##Problem Generation

For now we have opted to use a city in the form of a staggered grid as the representation of our problem.

In [ ]:

#dictionary is used because it would allow us to add more node classes in the future if needed and would be quicker to search for specific node type than if simply extending node class.
class Node():
  def __init__(self, edges = None, data=None):
    '''
    Args:
      edges: A list of edges connecting this node to neighbouring nodes.
      data: A dictionary containing specific information on the node.
      For example, if the node is a garage, data should contain parking price, garage capacity, current occupancy etc...
    '''
    if edges is None:
      self.edges = []
    else:
      self.edges = edges

    if data is None:
        self.data = {"node_type":"generic",
                    "x": 0,
                    "y": 0}
    else:
      self.data = data

  def __repr__(self):
      output_str = f"Edges: {self.edges} Data: {self.data}"
      return output_str
      # print(f"Neighbours: {self.neighbours[idx]} Weights:{self.weights[idx]}")

class Edge():
  def __init__(self, dest, min_time, current_time=None):
    self.dest = dest
    self.min_time = min_time
    if current_time is None:
      self.current_time = min_time
    else:
      self.current_time = current_time

  def __repr__(self):
    edge_string = f"Dest: {self.dest}, Min Travel Time: {self.min_time}, Current Travel Time: {self.current_time}"
    return edge_string


class Problem():
  def __init__(self, grid_size=None, min_size=None, max_size=None, grid_partitions = 2, min_capacity=20, max_capacity=100, min_density=0.5, max_density=5, min_price=1, max_price=10, transition_seed=None):
    # Generates a staggered grid to represent a city
    # More city generation options can potentially be implemented later, Voronoi based road generation could allow for more realistic city generation.

    if(grid_size is None):
      self.grid_size = 10
    else:
      self.grid_size = grid_size

    self.all_edges = []

    self.transition_seed = transition_seed

    self.random = random.Random(transition_seed);

    row_widths = [0]
    column_heights = [0]
    for value in range(grid_size):
      row_widths.append(row_widths[value] + random.randint(min_size,max_size))
      column_heights.append(column_heights[value] + random.randint(min_size,max_size))

    self.graph = []
    #A list of garages
    self.garages = []

    partition_size = (grid_size//grid_partitions)
    region_densities = []
    for value in range(grid_partitions**2):
      rand_density = random.randint(int(min_density*10),int(max_density*10))
      region_densities.append(rand_density)

    for node in range(grid_size**2):

      vertex = Node()
      self.graph.append(vertex)

    y_offset = random.randint(min_size, max_size)
    x_offset = random.randint(min_size, max_size)

    #Staggered Grid City Generation
    for node in range(grid_size**2):
      node_x_idx = (node)%grid_size
      node_y_idx = (node)//grid_size

      if((node+1) %grid_size ==0):
        last_column = True
      else:
        last_column = False
      if((node >= ((grid_size**2)-grid_size))):
        last_row = True
      else:
        last_row = False

      node_x = row_widths[node_x_idx]
      node_y = column_heights[node_y_idx]

      #Test for if not last column
      if(last_column==False):
        left_edge = Edge(node+1,((row_widths[node_x_idx+1])-row_widths[node_x_idx]))
        right_edge = Edge(node,((row_widths[node_x_idx+1])-row_widths[node_x_idx]))
        self.graph[node].edges.append(left_edge)
        self.graph[node+1].edges.append(right_edge)
        self.all_edges.append(left_edge)
        self.all_edges.append(right_edge)

      #Test for if not last row
      if(last_row==False):
        top_edge = Edge(node+grid_size,((column_heights[node_y_idx+1])-column_heights[node_y_idx]))
        bottom_edge = Edge(node,((column_heights[node_y_idx+1])-column_heights[node_y_idx]))
        self.graph[node].edges.append(top_edge)
        self.graph[node+grid_size].edges.append(bottom_edge)
        self.all_edges.append(top_edge)
        self.all_edges.append(bottom_edge)

      self.graph[node].data["x"] = node_x
      self.graph[node].data["y"] = node_y



      #Assign garage nodes based on region_density
      partition_region = node_x_idx//partition_size + node_y_idx//partition_size
      if(random.randint(0, 1000) < region_densities[partition_region]):
        garage_capacity = random.randint(min_capacity, max_capacity)
        self.garages.append(node)
        self.graph[node].data["node_type"] = "garage"
        self.graph[node].data["max_capacity"] = garage_capacity
        self.graph[node].data["current_capacity"] = random.randint(0, garage_capacity)
        self.graph[node].data["parking_price"] = random.randint(min_price, max_price)

  def __repr__(self):
    output_str = ""
    for node in range(len(self.graph)):
      output_str += f"Node {node}: {str(self.graph[node])}"
      output_str +="\n"
    return output_str


  def actions(self, state):
    '''
    Args:
      state: the current node the vehichle is located at.

    Returns:
      actions: an array of all possible actions for the current state.
      An action is an integer indicating the neighbouring nodes of the current state.
    '''
    return self.graph[state].edges

  def transition(self, do_print=False):
    '''
    Args:
    '''
    # DEBUGGING SET SEED LOGIC
    # if(do_print == True):
    #   print(self.random.randint(0,100))
    for garage in self.garages:
      # print(garage)
      current_capacity = self.graph[garage].data["current_capacity"]
      max_capacity = self.graph[garage].data["max_capacity"]
      current_price = self.graph[garage].data["parking_price"]
      new_capacity = self.random.randint(int(current_capacity*0.9), max_capacity)
      self.graph[garage].data["current_capacity"] = new_capacity
      new_price = self.random.randint(int((current_price*0.9)), int((current_price*1.1)))
      self.graph[garage].data["parking_price"] = new_price
      #randomize edge weights to signify traffic, travel time cannot go below minimum time. This signifies a speed limit, but also ensures A* remains admissable.
    modify_edges = self.random.sample(self.all_edges, k=int(len((self.all_edges))*0.1))

    # for node in self.graph:
    #   for edge in node.edges:
    for edge in modify_edges:
      multi = 0.1
      if(self.random.randint(0,1)==1):
        multi = -multi
      if(((1+multi)*edge.current_time) < edge.min_time):
        edge.current_time=edge.min_time
      else:
        edge.current_time = int((1+multi)*edge.current_time)
  def save_state(self):
    garage_data = [self.graph[garage].data for garage in self.garages]
    garage_data_copy = copy.deepcopy(garage_data)
    return garage_data_copy
  def reset_problem(self, stored_state):
    for garage_idx in range(len(self.garages)):
      for key in self.graph[self.garages[garage_idx]].data.keys():
        self.graph[self.garages[garage_idx]].data[key] = stored_state[garage_idx][key]
    for edge in self.all_edges:
      edge.current_time = edge.min_time
    self.random.seed(self.transition_seed)





testProblem = Problem(grid_size=10, min_size=12, max_size=20, grid_partitions=1,max_density=10,transition_seed=0)
print(testProblem.graph[testProblem.garages[0]])
# print(testProblem.graph[0].edges[0])
print(testProblem)
# testProblem.transition()
# print("_"*80)



IndexError: list index out of range

## A\* Solver Implementation

In [ ]:
'''
The purpose of the DPAUA agent is to solve a supplied problem assuming a given state.
The agent must store information such as how long it has take to reach the final garage, as well as the final price and walking distance of the garage.
We must adjust which garage we are aiming for based on changing garage prices.
A* will run at each timestep to determine the immediate next garage to check.
'''
class DPAUA_Agent():
  def __init__(self, problem, initial_state, dest_idx):
    self.initial_garage_state = problem.save_state()
    if initial_state is None:
      self.state = random.randrange(problem.grid_size**2)
    else:
      self.state = initial_state
    if problem is None:
      self.problem = None
    else:
      self.problem = problem
    self.dest_idx = dest_idx
    #Calculate manhattan distance heuristic for garages, this will be used as our walking distance assuming pedestrians aren't affected by traffic it should be perfectly accurate.
    self.h_score = []
    self.garage_distances = []
    if(len(self.garage_distances)<1):
      for garage in self.problem.garages:
        self.garage_distances.append(self.manhattan(garage, dest_idx))


  def manhattan(self, start_idx, dest_idx):
    start_node = self.problem.graph[start_idx]
    dest_node = self.problem.graph[dest_idx]
    return abs(start_node.data["x"]-dest_node.data["x"]) + abs(start_node.data["y"]-dest_node.data["y"])

  def aStar(self, start_idx, dest_idx):
    '''
    Parameters:

    Returns:


    takes in a starting state and attempts to calculate the optimal path from the starting state to the destination.
    Must precompute h_n for all nodes once per timestep to account for traffic.
    Must calculate g_n for each node explored.
    '''


    num_nodes = (self.problem.grid_size**2)
    state_node = self.problem.graph[start_idx]
    dest_node = self.problem.graph[dest_idx]
    g_score = [float('inf')]*num_nodes
    parent = [-1]*num_nodes
    frontier = []

    #Calculate manhattan distance heuristic for all nodes ONCE, if else is used in the case that in the future more complex logic may determine recalculating the heuristic.
    if(len(self.h_score)<1):
      for node in range(len(self.problem.graph)):
        self.h_score.append(self.manhattan(node, dest_idx))

    # Set initial g_score for starting node and add it to the priority queue.
    heapq.heappush(frontier,(self.h_score[start_idx],start_idx))
    g_score[start_idx] = 0

    #Main Loop: Continously explores graph until a path to the finish is found, or all nodes have been explored
    while frontier:
      #current is a tuple of f_score ,node index
      current = heapq.heappop(frontier)
      #return solution if final destination found otherwise keep searching
      if(current[1]==dest_idx):
        solution = []
        currsol_node = (dest_idx, None)
        while currsol_node != -1:
          solution.append((currsol_node))
          currsol_node=parent[currsol_node[0]]
        travel_time = g_score[dest_idx]
        return solution[::-1], travel_time
      #Search for path
      else:
        actions = self.problem.actions(current[1])
        for action in actions:
          neighbour_g = g_score[current[1]] + action.current_time
          if(neighbour_g < g_score[action.dest]):
            g_score[action.dest] = neighbour_g
            parent[action.dest] = ((current[1], action))
            heapq.heappush(frontier, (self.h_score[action.dest]+g_score[action.dest], action.dest))
    return -1 #No Solution

  def travel_time(self, path):
    '''
    returns:
      travel_time: the travel time of a path provided in the form of a list of edges
    '''
    travel_time = 0
    for edge in path:
      if(edge[1] is not None):
        travel_time+=edge[1].current_time
    return travel_time

  def find_solution(self, max_w, visualise = False):
    '''
    Parameters:
      self
      state:
        The starting location of the vehichle
      destination:
        The target node of the passenger
      heuristic:
        The heuristic to use for the algorithm.
        For now, we will simply use manhattan distance, but later euclidean distance could be tested for more complex "cities"
    Returns:
      Solution:
        A dictionary containing the path of the final solution, the time taken to reach the final destination, the final price of the garage chosen, as well as the walking distance to the destination.
    '''
    # We must run a loop which decides which turn to take at each step and tracks the time taken for each node.
    # We should only consider rerunning A* if the estimated travel time of our current path has changed significantly since beginning our journey.

    #TODO: implement performance calculation and use of A* and garage pruning to find final solution

    #Prune garages based on walking distance to destination and pick best garage to pathfind to based on parking price, and current capacity
    self.problem.reset_problem(self.initial_garage_state)
    valid_garages = []
    min_cost = float('inf')
    min_garage_id = None
    for garage in range(len(self.problem.garages)):

      if(self.garage_distances[garage]) < max_w:
        # overall cost will be calculated based on walking distance from garage to destination, manhattan distance (rough estimated travel time from current state to garage), and garage price
        est_cost = self.manhattan(self.problem.garages[garage], self.dest_idx)*5 + self.manhattan(self.state, self.problem.garages[garage]) + self.problem.graph[self.problem.garages[garage]].data["parking_price"]
        if(est_cost < min_cost):
          min_cost = est_cost
          min_garage_id = garage
        valid_garages.append((self.problem.garages[garage], est_cost))
    min_garage = self.problem.garages[min_garage_id]
    # print(f"valid_garages: {valid_garages}")


    current_path, est_travel = self.aStar(self.state, min_garage)
    final_path = []
    final_travel_time = 0
    curr = 0
    while(self.state != min_garage):
      if(self.travel_time(current_path[curr:]) > (1.4 * (est_travel))):
        curr = 0
        current_path, est_travel = self.aStar(self.state, min_garage)
      final_path.append(current_path[curr])
      if(current_path[curr][1] is not None):
        final_travel_time+=current_path[curr][1].current_time
      self.state = current_path[curr][0]

      curr+=1
      self.problem.transition(True)

    # print(self.problem.graph[min_garage])
    performance = {
        "travel_time":final_travel_time,
        "cost":self.problem.graph[min_garage].data["parking_price"],
        "walking_distance":self.manhattan(min_garage, self.dest_idx)
    }
    return final_path, performance






##Q-learning Solution Implementation

In [ ]:
class QLearningAgent:
    def __init__(self, problem, start_idx, dest_idx, max_walk, alpha=0.1, gamma=0.90, epsilon=0.95):
        # Store reference to the environment (graph, garages, etc.)
        self.problem = problem

        self.initial_state = self.problem.save_state()

        # Starting node
        self.start_idx = start_idx

        # Destination node (used to compute walking distance)
        self.dest_idx = dest_idx

        # Maximum allowed walking distance
        self.max_walk = max_walk

        # Learning rate (how fast Q-values update)
        self.alpha = alpha

        # Discount factor (importance of future rewards)
        self.gamma = gamma

        # Exploration rate (probability of choosing random action)
        self.epsilon = epsilon

        # Q-table: dictionary storing Q(state, action)
        self.Q = {}

        # Initialize Q-values for all state-action pairs to 0
        for state in range(len(problem.graph)):
            self.Q[state] = {}

            # Get all possible actions (edges) from this state
            actions = self.problem.actions(state)

            # Store Q-value for each possible next node

            for action in actions:
                self.Q[state][action.dest] = 0.0
                # if(self.problem.graph[action.dest].data["node_type"] == "garage"):
                #   walking_distance = self.manhattan(action.dest,self.dest_idx)
                #   parking_cost = self.problem.graph[action.dest].data["parking_price"]
                #   if(walking_distance > max_walk):
                #     self.Q[state][action.dest] -= 500
                #   else:
                #     self.Q[state][action.dest] += (2*max_walk - 2*parking_cost - walking_distance)

    def manhattan(self, start_idx, dest_idx):
        # Compute Manhattan distance between two nodes
        start_node = self.problem.graph[start_idx]
        dest_node = self.problem.graph[dest_idx]

        return abs(start_node.data["x"]-dest_node.data["x"]) + abs(start_node.data["y"]-dest_node.data["y"])

    def get_edge(self, state, action_dest):
        # Find the edge connecting state -> action_dest
        for edge in self.problem.graph[state].edges:
            if edge.dest == action_dest:
                return edge
        return None

    def get_actions(self, state):
        # Returns list of Edge objects from current state
        actions = self.problem.actions(state)
        return self.problem.actions(state)

    def choose_action(self, state):
        # Get possible actions from current state
        actions = self.get_actions(state)

        # If no available actions, return None
        if not actions:
            return None

        # Epsilon-greedy strategy:
        # With probability epsilon -> explore (random action)
        if random.random() < self.epsilon:
            return random.choice(actions).dest

        # Otherwise -> exploit (choose best Q-value action)
        return max(actions, key=lambda e: self.Q[state][e.dest]).dest

    def get_reward(self, state, action, next_state, done):
        # Get edge used for this transition
        edge = self.get_edge(state, action)

        # Penalize travel time
        step_reward = -edge.current_time

        distance_penalty = 0.1*(self.manhattan(state, self.dest_idx) - self.manhattan(action, self.dest_idx))

        # If reached terminal state (garage)
        if done:
            node = self.problem.graph[action]

            # Get garage properties
            parking_price = node.data.get("parking_price", 0)
            walking_distance = self.manhattan(action, self.dest_idx)
            curr_capacity = node.data.get("current_capacity", 0)
            max_capacity = node.data.get("max_capacity", 0)

            # Penalize invalid garages
            if walking_distance > self.max_walk:
                return step_reward + distance_penalty -10000

            if curr_capacity >= max_capacity:
                return step_reward + distance_penalty -300

            # Reward valid garages
            return step_reward + distance_penalty + 3000 - (3 * parking_price) - 20*walking_distance

        # Otherwise return step penalty
        return step_reward + distance_penalty

    def transition_qlearning(self, state, action):
        # Action directly leads to next node
        next_state = action

        # Check if next state is a garage (terminal state)
        done = next_state in self.problem.garages
        self.problem.transition()

        return next_state, done

    def train(self, episodes, max_steps=300):
        # Train the agent over multiple episodes
        epsilon_increment = self.epsilon*(1/episodes)
        for _ in range(episodes):
            state = self.start_idx
            done = False
            steps = 0

            # Run episode until terminal state or max steps reached
            while not done and steps < max_steps:
                # Select action using epsilon-greedy policy
                # Action is the next node to travel to
                action = self.choose_action(state)

                if action is None:
                    break

                next_state, done = self.transition_qlearning(state, action)
                reward = self.get_reward(state, action, next_state, done)

                if done:
                    # Terminal state: Future value is exactly ZERO
                    target = reward
                else:
                    next_actions = self.get_actions(next_state)
                    max_next_q = max((self.Q[next_state][e.dest] for e in next_actions), default=0)
                    target = reward + self.gamma * max_next_q

                # Update Q-value
                self.Q[state][action] += self.alpha * (target - self.Q[state][action])

                # Move to next state
                state = next_state
                steps += 1
            self.epsilon-=epsilon_increment

    def get_best_path(self, problem, max_steps=50):
        # Extract the best path using learned Q-values
        self.problem.reset_problem(self.initial_state)
        travel_time = 0
        state = self.start_idx
        path = [state]
        # qBestPath =[self.Q[state]]

        for _ in range(max_steps):
            # Get possible actions
            actions = self.get_actions(state)

            if not actions:
                break

            # Choose best action (highest Q-value)
            best_action = max(actions, key=lambda e: self.Q[state][e.dest]).dest
            travel_time += max(actions, key=lambda e: self.Q[state][e.dest]).current_time

            # Move to next state
            path.append(best_action)
            # qBestPath.append((best_action,self.Q[best_action]))
            state = best_action
            self.problem.transition(True)

            # Stop if we reach a garage
            if state in self.problem.garages:
                break
        if path[-1] in self.problem.garages:
          performance = {
          "travel_time":travel_time,
          "cost":self.problem.graph[path[-1]].data["parking_price"],
          "walking_distance":self.manhattan(path[-1], self.dest_idx)
          }
        else: performance = -1
        return path, performance

##Testing
The Solver Uses A\* to find the shortest path to a chosen garage based on the factors of the problem. In the end we get a final output including the travel time to the garage, the walking distance from the destination, and the cost of parking. Currently these factors are considered in a somewhat skewed manner, but in the future we could consider including a utility profile to determine the choice of garage. We also recalculate The path of A\* if traffic changes the expected travel time by more than a certain amount. For now this is hard coded as predicted travel time increasing by more than 40%. With the way we've represented garage pricing and capacity, in the future we could take those into account when calculating the optimal path, but due to time limitations, this is currently unimplemented.

In [ ]:
# A* Test
testProblem = Problem(grid_size=30, min_size=12, max_size=20, grid_partitions=6,max_density=10,transition_seed=0)
astar_agent = DPAUA_Agent(testProblem, 42, 76)
# path, travel_time = astar_agent.aStar(300,488)
# print(path)
best_path, performance = astar_agent.find_solution(max_w = 150)
print(f"final_path:\n {best_path}\nfinal_performance:\n{performance}")


# Q-learning test
# testProblem = Problem(grid_size=30, min_size=12, max_size=20, grid_partitions=6,max_density=10,transition_seed=0)
# ql_agent = QLearningAgent(testProblem, start_idx=458, dest_idx=725, max_walk=150)
# ql_agent.train(episodes=2000)





In [ ]:
# best_path, performance = ql_agent.get_best_path(300)

# print("Learned best path:", best_path)

# if best_path and best_path[-1] in testProblem.garages:
#     final_node = testProblem.graph[best_path[-1]]
#     print("Parking Price:", performance['cost'])
#     print("Current Capacity:", final_node.data.get("current_capacity"))
#     print("Max Capacity:", final_node.data.get("max_capacity"))
#     print("Walking Distance:", performance['walking_distance'])
#     print("Travel Time:", performance['travel_time'])
# else:
#     print("Path did not end at a garage.")

## Route visualization

In [ ]:
#TODO: Visualisation using matplotlib


# ASTAR GRAPHING

# visualise_graph(testProblem)

#def visualise_graph(problem, final_path=None, start_idx=None, dest_idx=None, performance=None):
 #import matplotlib.pyplot as plt
  #from matplotlib.collections import LineCollection

  # Collect node coordinates.
 # x_coords = [node.data["x"] for node in problem.graph]
 # y_coords = [node.data["y"] for node in problem.graph]

  # Build road segments once, avoiding duplicate undirected edges.
  #road_segments = []

  #seen_edges = set()
  #for src_idx, node in enumerate(problem.graph):
   # for edge in node.edges:
     # edge_key = tuple(sorted((src_idx, edge.dest)))
     # if edge_key in seen_edges:
        #continue
     # seen_edges.add(edge_key)
      #src = problem.graph[src_idx].data
    #  dst = problem.graph[edge.dest].data
      #road_segments.append([(src["x"], src["y"]), (dst["x"], dst["y"])])

  #fig, ax = plt.subplots(figsize=(16, 10))
 # ax.add_collection(LineCollection(road_segments, colors="#c9ced6", linewidths=0.8, zorder=1))

  # Plot all nodes and garages.
 # ax.scatter(x_coords, y_coords, s=8, c="#4b5563", alpha=0.55, label="Road Nodes", zorder=2)
 # garage_x = [problem.graph[g].data["x"] for g in problem.garages]
 # garage_y = [problem.graph[g].data["y"] for g in problem.garages]
 # ax.scatter(garage_x, garage_y, s=26, c="#e74c3c", marker="s", label="Garages", zorder=4)

  #chosen_garage = None
 # if final_path:
  #  path_nodes = [step[0] for step in final_path]
   # path_segments = []
    #for idx in range(len(path_nodes) - 1):
     # n1 = problem.graph[path_nodes[idx]].data
    #  n2 = problem.graph[path_nodes[idx + 1]].data
      #path_segments.append([(n1["x"], n1["y"]), (n2["x"], n2["y"])])
   # if path_segments:
    #  ax.add_collection(LineCollection(path_segments, colors="#1f77b4", linewidths=2.8, zorder=5, label="Chosen Path"))
  #  chosen_garage = path_nodes[-1]

 # if start_idx is not None:
   # start_node = problem.graph[start_idx].data
   # ax.scatter(start_node["x"], start_node["y"], s=170, c="#2ecc71", marker="*", edgecolors="black", linewidths=0.8, label="Start", zorder=7)

  #if dest_idx is not None:
   # dest_node = problem.graph[dest_idx].data
  #  ax.scatter(dest_node["x"], dest_node["y"], s=170, c="#f1c40f", marker="X", edgecolors="black", linewidths=0.8, label="Destination", zorder=7)

 # if chosen_garage is not None and chosen_garage in problem.garages:
   # garage_data = problem.graph[chosen_garage].data
   # gx = garage_data["x"]
   # gy = garage_data["y"]
   # ax.scatter(gx, gy, s=180, facecolors="none", edgecolors="#0b3d91", linewidths=2, label="Chosen Garage", zorder=8)
   # stats_lines = [
    #  f"Garage ID: {chosen_garage}",
     # f"Price: {garage_data.get('parking_price', 'N/A')}",
     # f"Capacity: {garage_data.get('current_capacity', 'N/A')}/{garage_data.get('max_capacity', 'N/A')}",
    #]
   # if isinstance(performance, dict):
    #  stats_lines.append(f"Travel Time: {performance.get('travel_time', 'N/A')}")
    #  stats_lines.append(f"Walking Distance: {performance.get('walking_distance', 'N/A')}")
   # ax.text(
     # 0.02, 0.98, "\n".join(stats_lines),
     # transform=ax.transAxes,
     # ha="left", va="top",
     # fontsize=10,
     # bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.9, edgecolor="#b0b0b0"),
    #  zorder=10
   # )

  #ax.set_title("DPAUA Route Visualisation")
  #ax.set_xlabel("X")
  #ax.set_ylabel("Y")
 # ax.grid(alpha=0.2)
 # ax.autoscale()
  #ax.set_aspect("equal", adjustable="box")

 # handles, labels = ax.get_legend_handles_labels()
 # uniq = dict(zip(labels, handles))
  #ax.legend(uniq.values(), uniq.keys(), loc="upper right", framealpha=0.95)

  #plt.tight_layout()
 # plt.show()


#visualise_graph(testProblem, final_path=final_path, start_idx=300, dest_idx=488, performance=performance)

def visualise_graph(problem, final_path=None, start_idx=None, dest_idx=None, performance=None):
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection

    # Collect node coordinates
    x_coords = [node.data["x"] for node in problem.graph]
    y_coords = [node.data["y"] for node in problem.graph]

    # Build road segments once, avoiding duplicate undirected edges
    road_segments = []
    seen_edges = set()

    for src_idx, node in enumerate(problem.graph):
        for edge in node.edges:
            edge_key = tuple(sorted((src_idx, edge.dest)))
            if edge_key in seen_edges:
                continue
            seen_edges.add(edge_key)

            src = problem.graph[src_idx].data
            dst = problem.graph[edge.dest].data
            road_segments.append([(src["x"], src["y"]), (dst["x"], dst["y"])])

    fig, ax = plt.subplots(figsize=(16, 10))
    ax.add_collection(LineCollection(road_segments, colors="#c9ced6", linewidths=0.8, zorder=1))

    # Plot all nodes
    ax.scatter(x_coords, y_coords, s=8, c="#4b5563", alpha=0.55, label="Road Nodes", zorder=2)

    # Plot garages
    garage_x = [problem.graph[g].data["x"] for g in problem.garages]
    garage_y = [problem.graph[g].data["y"] for g in problem.garages]
    ax.scatter(garage_x, garage_y, s=26, c="#e74c3c", marker="s", label="Garages", zorder=4)

    chosen_garage = None

    if final_path:
        # Handle both path formats:
        # A* path -> [(node, edge), ...]
        # Q-learning path -> [node, node, node, ...]
        if isinstance(final_path[0], tuple):
            path_nodes = [step[0] for step in final_path]
        else:
            path_nodes = final_path

        path_segments = []
        for idx in range(len(path_nodes) - 1):
            n1 = problem.graph[path_nodes[idx]].data
            n2 = problem.graph[path_nodes[idx + 1]].data
            path_segments.append([(n1["x"], n1["y"]), (n2["x"], n2["y"])])

        if path_segments:
            ax.add_collection(
                LineCollection(path_segments, colors="#1f77b4", linewidths=2.8, zorder=5, label="Chosen Path")
            )

        chosen_garage = path_nodes[-1]

    # Plot start node
    if start_idx is not None:
        start_node = problem.graph[start_idx].data
        ax.scatter(
            start_node["x"], start_node["y"],
            s=170, c="#2ecc71", marker="*", edgecolors="black",
            linewidths=0.8, label="Start", zorder=7
        )

    # Plot destination node
    if dest_idx is not None:
        dest_node = problem.graph[dest_idx].data
        ax.scatter(
            dest_node["x"], dest_node["y"],
            s=170, c="#f1c40f", marker="X", edgecolors="black",
            linewidths=0.8, label="Destination", zorder=7
        )

    # Highlight chosen garage and show stats
    if chosen_garage is not None and chosen_garage in problem.garages:
        garage_data = problem.graph[chosen_garage].data
        gx = garage_data["x"]
        gy = garage_data["y"]

        ax.scatter(
            gx, gy,
            s=180, facecolors="none", edgecolors="#0b3d91",
            linewidths=2, label="Chosen Garage", zorder=8
        )

        stats_lines = [
            f"Garage ID: {chosen_garage}",
            f"Price: {garage_data.get('parking_price', 'N/A')}",
            f"Capacity: {garage_data.get('current_capacity', 'N/A')}/{garage_data.get('max_capacity', 'N/A')}",
        ]

        if isinstance(performance, dict):
            stats_lines.append(f"Travel Time: {performance.get('travel_time', 'N/A')}")
            stats_lines.append(f"Walking Distance: {performance.get('walking_distance', 'N/A')}")

        ax.text(
            0.02, 0.98,
            "\n".join(stats_lines),
            transform=ax.transAxes,
            ha="left", va="top",
            fontsize=10,
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.9, edgecolor="#b0b0b0"),
            zorder=10
        )

    ax.set_title("DPAUA Route Visualisation")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.grid(alpha=0.2)
    ax.autoscale()
    ax.set_aspect("equal", adjustable="box")

    handles, labels = ax.get_legend_handles_labels()
    uniq = dict(zip(labels, handles))
    ax.legend(uniq.values(), uniq.keys(), loc="upper right", framealpha=0.95)

    plt.tight_layout()
    plt.show()

#     ql_performance = None
# if best_path and best_path[-1] in testProblem.garages:
#     final_node = testProblem.graph[best_path[-1]]
#     ql_performance = {
#         "travel_time": len(best_path) - 1,
#         "walking_distance": ql_agent.manhattan(best_path[-1], ql_agent.dest_idx)
#     }

visualise_graph(
    testProblem,
    final_path=best_path,
    start_idx=42,
    dest_idx=76,
    performance=performance
)


##Comparison of Techniques

In [ ]:
def find_non_garage(problem):
  num_nodes = len(problem.graph)
  selected_node = random.randrange(num_nodes)

  while(selected_node in problem.garages):
    selected_node = random.randrange(num_nodes)
  return selected_node

# Initialize Performance Metrics
# tt = travel time, wd = walking distance, pc = parking cost
failures_astar    = 0
total_tt_astar    = 0
total_pc_astar    = 0
total_wd_astar    = 0
total_wins_astar  = 0

failures_ql       = 0
total_tt_ql       = 0
total_pc_ql       = 0
total_wd_ql       = 0
total_wins_ql     = 0

successful_trials = 0
draws             = 0

# Testing Loop:
for seed in range(0, 50):
  #initialize per problem performance metrics
  tt_astar    = 0
  pc_astar    = 0
  wd_astar    = 0

  tt_ql       = 0
  pc_ql       = 0
  wd_ql       = 0

  # Determine a valid starting and ending position (start and end are not a garage)


  # Setup A* Agent to solve problem with set seed

  random.seed(seed)
  problem = Problem(grid_size=30, min_size=12, max_size=20, grid_partitions=6,max_density=10,transition_seed=seed)
  start_pos = find_non_garage(problem)
  end_pos = find_non_garage(problem)
  print(f"Problem #{seed+1} | start_pos: {start_pos} | end_pos: {end_pos}\n")
  astar_agent = DPAUA_Agent(problem, start_pos, end_pos)

  # Test A* Agent
  best_path, performance = astar_agent.find_solution(max_w=150)
  # visualise_graph(
  #   problem,
  #   final_path=best_path,
  #   start_idx=start_pos,
  #   dest_idx=end_pos,
  #   performance=performance
  # )
  print(f"A* Performance:\n{performance}\n")
  tt_astar, pc_astar, wd_astar  =  performance.values()

  # Setup QLearning Agent
  random.seed(seed)
  problem = Problem(grid_size=30, min_size=12, max_size=20, grid_partitions=6,max_density=10,transition_seed=seed)
  start_pos = find_non_garage(problem)
  end_pos = find_non_garage(problem)
  ql_agent = QLearningAgent(problem, start_idx= start_pos, dest_idx=end_pos, max_walk=150)

  # Test QLearning
  ql_agent.train(episodes=2000)
  best_path, performance = ql_agent.get_best_path(300)

  # visualise_graph(
  #   problem,
  #   final_path=best_path,
  #   start_idx=start_pos,
  #   dest_idx=end_pos,
  #   performance=performance
  # )
  print(f"QL Performance:\n{performance}\n")
  if (performance != -1):
    tt_ql, pc_ql, wd_ql  =  performance.values()
    total_tt_ql       += tt_ql
    total_pc_ql       += pc_ql
    total_wd_ql       += wd_ql
    total_tt_astar    += tt_astar
    total_pc_astar    += pc_astar
    total_wd_astar    += wd_astar
    successful_trials += 1
  else:
    failures_ql +=1





  combined_astar = tt_astar + pc_astar + wd_astar
  combined_ql = tt_ql + pc_ql + wd_ql
  if(combined_astar < combined_ql):
    total_wins_astar += 1
  elif(combined_ql < combined_astar):
    total_wins_ql += 1
  else:
    draws += 1

print(f"Total Wins A*: {total_wins_astar} | Total Wins QL: {total_wins_ql} | Draws: {draws}")
print("-"*40)
print(f"Performance For A* Over {successful_trials} Problems")
print(f"Total Travel Time: {total_tt_astar} | Total Parking Costs: {total_pc_astar} | Total Walking Distance: {total_wd_astar}")
print("-"*40)
print(f"Performance For QL* Over {successful_trials} Problems")
print(f"Total Travel Time: {total_tt_ql} | Total Parking Costs: {total_pc_ql} | Total Walking Distance: {total_wd_ql} | Failures: {failures_ql}")